# Valores faltantes: cómo identificarlos

En un dataset real es normal encontrar celdas sin información. El hecho de que un valor esté ausente no explica por sí solo qué ocurrió ni cuál es la mejor solución. Puede tratarse de un dato que no se registró, de una variable que no aplicaba, de un marcador textual o de un problema ocurrido durante la carga.

En este capítulo aprenderemos a localizar y medir esas ausencias antes de tomar decisiones. La meta es construir un diagnóstico que permita evaluar su importancia y elegir un tratamiento con fundamento.

Al finalizar podremos:

- reconocer cómo representa Pandas la información faltante;
- contar ausencias por fila y por columna;
- interpretar su proporción respecto del tamaño de la tabla;
- localizar registros afectados;
- distinguir `NaN` de textos como `UNKNOWN` o `ERROR`;
- documentar hallazgos sin eliminar datos prematuramente.

### Criterio de interpretación

Esta primera revisión establece una referencia para las comprobaciones posteriores. Mantener claro el punto de partida facilita comparar qué cambió y qué problemas permanecen después de limpiar.

## Punto de partida

En el ejercicio anterior revisamos la idea general de preparar datos: primero se observa la fuente, después se identifican problemas y finalmente se eligen transformaciones que puedan comprobarse.

Ahora concentraremos ese proceso en una pregunta concreta: ¿dónde falta información y qué tan importante es? Antes de completar, borrar o conservar una celda, necesitamos saber cómo aparece el problema, cuántos registros alcanza y qué variables están involucradas.

La exploración se realizará con Pandas y se apoyará en conteos, porcentajes, ejemplos de filas y revisiones de valores especiales.

### Criterio de interpretación

La ausencia debe interpretarse junto con el significado de la variable. La misma cantidad de valores faltantes puede ser tolerable en una columna descriptiva y crítica en una variable necesaria para calcular resultados.

## Fuente de práctica

Usaremos el dataset **Cafe Sales — Dirty Data for Cleaning Training**. Cada fila representa una venta y contiene variables como el artículo, la cantidad, el precio unitario, el importe total, el método de pago, la ubicación y la fecha.

El usuario deberá cargar el archivo directamente en Google Colab. Esto permite repetir el notebook con el CSV disponible localmente, sin depender de una descarga automática ni de una ruta específica del equipo.

La fuente contiene situaciones útiles para practicar: celdas nulas, valores especiales escritos como texto y columnas que deben revisarse antes de utilizarlas en cálculos. En esta etapa únicamente diagnosticaremos; las decisiones de corrección quedarán para el siguiente capítulo.

### Criterio de interpretación

En una fuente real, el formato de almacenamiento y el significado del dato no siempre coinciden. Por eso conviene observar ejemplos concretos antes de convertir, reemplazar o eliminar valores.

In [3]:
import io
import pandas as pd
from google.colab import files

# El usuario selecciona el archivo desde su equipo.
uploaded = files.upload()
archivos = list(uploaded.keys())
archivos_csv = [archivo for archivo in archivos if archivo.lower().endswith(".csv")]

if not archivos_csv:
    raise ValueError("Debes subir al menos un archivo con extensión .csv")

nombre_csv = archivos_csv[0]
df = pd.read_csv(io.BytesIO(uploaded[nombre_csv]), sep=';')

df.head()

Saving dirty_cafe_sales.csv to dirty_cafe_sales (2).csv


,Warning: truncated output (original token count: 137577)
0,Total output lines: 10002
1,"Transaction ID,Item,Quantity,Price Per Unit,To..."
2,"TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takea..."
3,"TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023..."
4,"TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-..."


Una vista inicial de `head()` permite comprobar que el archivo se leyó y que el objeto `df` contiene registros.

No debemos confundir esta comprobación con una validación completa. Las primeras filas pueden no incluir los casos más interesantes, como una ausencia poco frecuente o una categoría utilizada solo en una parte de la tabla.

Por eso continuaremos con preguntas más específicas sobre la presencia y distribución de los valores faltantes.

### Criterio de interpretación

El diagnóstico combina evidencia local y global: las filas muestran casos reales, mientras que los conteos y porcentajes permiten evaluar si esos casos son aislados o forman un patrón.

## Interpretar una ausencia

Un dato faltante es una posición en la que no tenemos información utilizable. Esta descripción es sencilla, pero sus causas pueden ser muy distintas: el usuario pudo dejar un campo vacío, el sistema pudo omitirlo o el valor pudo perderse durante la exportación.

La ausencia tampoco tiene la misma importancia en todas las variables. Un método de pago desconocido puede conservarse como una categoría pendiente, mientras que la falta de cantidad o precio puede impedir calcular el total de una venta.

Por esa razón, el primer objetivo no es corregir. Es identificar la ausencia, entender su contexto y medir el impacto potencial sobre el análisis.

### Criterio de interpretación

Todavía no se aplica una estrategia de limpieza. Separar medición y transformación ayuda a evitar decisiones irreversibles basadas en una sola muestra o en una interpretación apresurada.

## Representación en Pandas

Pandas suele utilizar `NaN` para señalar que una celda numérica o de otro tipo no contiene un valor disponible. También puede trabajar con otras representaciones nulas dependiendo del tipo de dato y de cómo se importó el archivo.

Lo importante es que `NaN` no debe confundirse con el texto `"NaN"` ni con cadenas como `"UNKNOWN"`. El primero es una marca reconocida por Pandas; las segundas pueden comportarse como categorías ordinarias hasta que decidamos interpretarlas.

Conocer esta diferencia evita que un conteo de faltantes sea menor de lo real.

### Criterio de interpretación

Estas comprobaciones también son útiles para comunicar resultados. Una tabla de diagnóstico permite explicar con claridad qué se encontró, dónde apareció y qué información adicional se necesita.

## Primera detección

El método `isna()` recorre el `DataFrame` y devuelve una máscara booleana. En esa máscara, `True` indica que Pandas reconoce una ausencia y `False` indica que encontró algún valor.

Esta herramienta es útil para localizar el problema, aunque su resultado conserva exactamente las dimensiones de la tabla original. Para interpretar mejor la información necesitaremos resumir esa máscara en los siguientes pasos.

### Criterio de interpretación

Cuando una ausencia se concentra en una variable central, conviene estudiar sus consecuencias antes de continuar con cálculos o modelos. La calidad del resultado depende de reconocer esas limitaciones.

In [ ]:
df.isna()

La salida conserva el mismo número de filas y columnas que `df`, pero reemplaza cada dato por una marca lógica. De esta forma se puede observar la posición de cada ausencia.

En una tabla pequeña esta vista puede ser suficiente para revisar ejemplos. En una fuente más grande, conviene agregar los valores por columna o por fila para obtener medidas más fáciles de interpretar.

### Criterio de interpretación

La revisión de valores especiales complementa el análisis de `NaN`. Un dataset puede representar la falta de información de varias maneras, especialmente cuando proviene de sistemas distintos o de procesos de captura manual.

## Conteo por variable

Para convertir la máscara en un diagnóstico resumido, combinaremos `isna()` con `sum()`. En Python, cada `True` se cuenta como uno y cada `False` como cero, por lo que la suma indica cuántas celdas ausentes hay en cada columna.

Este conteo permite separar las variables completas de aquellas que necesitan una revisión posterior. Todavía no indica qué tratamiento aplicar; únicamente muestra la magnitud absoluta del problema.

### Criterio de interpretación

Ordenar y resumir las evidencias no modifica el contenido de la fuente. Solo mejora la lectura y ayuda a establecer un orden razonable para las siguientes decisiones.

In [ ]:
df.isna().sum()

El resultado permite localizar rápidamente las columnas con información incompleta.

Una columna con cero ausencias no requiere el mismo análisis que otra con varios registros afectados, aunque el número por sí solo no sea suficiente para juzgar la calidad. También debemos considerar el papel de la variable y el tamaño total del dataset.

Este será nuestro primer registro cuantitativo del diagnóstico.

### Criterio de interpretación

La validación posterior deberá responder a la señal encontrada. Si se modifican nulos, se revisarán nulos restantes; si se normalizan textos, se volverán a contar las categorías; si se eliminan filas, se compararán las dimensiones.

## Poner las ausencias en proporción

El número de celdas vacías debe interpretarse respecto del total de filas. Cinco ausencias pueden ser relevantes en una tabla de veinte registros y casi irrelevantes en una tabla de diez mil.

Para comparar columnas de forma justa calcularemos el porcentaje de valores faltantes. Esta medida no reemplaza al conteo absoluto, pero añade una perspectiva relativa que ayuda a establecer prioridades.

### Criterio de interpretación

Esta primera revisión establece una referencia para las comprobaciones posteriores. Mantener claro el punto de partida facilita comparar qué cambió y qué problemas permanecen después de limpiar.

In [ ]:
porcentaje_faltantes = df.isna().mean() * 100

porcentaje_faltantes

El porcentaje indica qué parte de cada columna no contiene un dato reconocido.

Un valor cercano a cero sugiere una ausencia puntual; un porcentaje elevado señala que la variable podría requerir una estrategia específica o incluso una revisión de su utilidad. La decisión final dependerá del contexto y del análisis que se quiera realizar.

Por ahora utilizaremos esta métrica para describir el problema, no para eliminar registros automáticamente.

### Criterio de interpretación

La ausencia debe interpretarse junto con el significado de la variable. La misma cantidad de valores faltantes puede ser tolerable en una columna descriptiva y crítica en una variable necesaria para calcular resultados.

In [ ]:
diagnostico_faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": df.isna().mean() * 100
})

diagnostico_faltantes

La tabla combina dos perspectivas: cuántas celdas faltan y qué proporción representan.

Esta combinación ayuda a priorizar la investigación. El conteo permite conocer el número de casos concretos y el porcentaje permite comparar columnas con diferentes niveles de completitud.

Al documentar ambos valores, otra persona podrá entender mejor por qué una variable fue considerada de mayor o menor prioridad.

### Criterio de interpretación

En una fuente real, el formato de almacenamiento y el significado del dato no siempre coinciden. Por eso conviene observar ejemplos concretos antes de convertir, reemplazar o eliminar valores.

## Ordenar para priorizar

Cuando hay varias columnas, leer el diagnóstico en su orden original puede ocultar dónde se concentra el problema. Ordenar la tabla de mayor a menor cantidad de ausencias coloca primero los casos que requieren más atención.

La ordenación no cambia los datos ni resuelve los faltantes; solo mejora la lectura del diagnóstico y ayuda a decidir qué revisar primero.

### Criterio de interpretación

El diagnóstico combina evidencia local y global: las filas muestran casos reales, mientras que los conteos y porcentajes permiten evaluar si esos casos son aislados o forman un patrón.

In [ ]:
diagnostico_faltantes.sort_values("faltantes", ascending=False)

Al mostrar primero las columnas con más ausencias, se vuelve más sencillo identificar los principales focos de revisión.

Esta presentación puede ser útil al comunicar hallazgos, pero no debe interpretarse como una regla automática. Una variable con pocos faltantes también puede ser crítica si participa en un cálculo esencial.

### Criterio de interpretación

Todavía no se aplica una estrategia de limpieza. Separar medición y transformación ayuda a evitar decisiones irreversibles basadas en una sola muestra o en una interpretación apresurada.

In [ ]:
diagnostico_faltantes.sort_values("porcentaje", ascending=False)

El porcentaje se calcula utilizando el mismo total de filas, por lo que suele producir un orden parecido al conteo cuando todas las columnas tienen igual número de registros.

Aun así, expresarlo en términos porcentuales hace que el resultado sea más fácil de explicar y comparar. En un análisis posterior, ambas medidas pueden servir para justificar una decisión de limpieza.

### Criterio de interpretación

Estas comprobaciones también son útiles para comunicar resultados. Una tabla de diagnóstico permite explicar con claridad qué se encontró, dónde apareció y qué información adicional se necesita.

## Examinar los registros afectados

Los resúmenes indican dónde está el problema, pero no muestran cómo se presenta en las filas reales. Para comprender el contexto, construiremos una máscara que identifique cualquier registro que tenga al menos un valor faltante.

Así podremos observar ejemplos concretos y comprobar si las ausencias se concentran en ciertas variables o aparecen junto con otros valores especiales.

### Criterio de interpretación

Cuando una ausencia se concentra en una variable central, conviene estudiar sus consecuencias antes de continuar con cálculos o modelos. La calidad del resultado depende de reconocer esas limitaciones.

In [ ]:
filas_con_faltantes = df.isna().any(axis=1)

filas_con_faltantes

`df.isna()` genera una marca por celda. Con `.any(axis=1)` preguntamos si existe al menos un `True` en cada fila; el resultado es una serie booleana que permite filtrar los registros afectados.

Esta combinación es una forma flexible de pasar de un diagnóstico por celda a una selección de filas. El parámetro `axis=1` es importante porque indica que la revisión se realiza horizontalmente, fila por fila.

### Criterio de interpretación

La revisión de valores especiales complementa el análisis de `NaN`. Un dataset puede representar la falta de información de varias maneras, especialmente cuando proviene de sistemas distintos o de procesos de captura manual.

In [ ]:
df[filas_con_faltantes].head(10)

La muestra permite observar qué variables están vacías dentro de registros concretos.

Los ejemplos ayudan a formular hipótesis: quizá varias columnas queden vacías en la misma transacción, quizá la ausencia esté aislada o quizá se relacione con una categoría particular. Estas hipótesis deberán comprobarse con conteos y no solo con una muestra.

### Criterio de interpretación

Ordenar y resumir las evidencias no modifica el contenido de la fuente. Solo mejora la lectura y ayuda a establecer un orden razonable para las siguientes decisiones.

## Variables con mayor impacto

No todas las columnas aportan el mismo valor al análisis. En una tabla de ventas, el artículo, la cantidad, el precio, el importe y la fecha suelen ser importantes para describir operaciones y calcular métricas.

Por eso revisaremos de manera enfocada las columnas centrales. Este filtro no elimina el resto de la información; simplemente permite examinar primero las variables cuya ausencia podría afectar más las conclusiones.

### Criterio de interpretación

La validación posterior deberá responder a la señal encontrada. Si se modifican nulos, se revisarán nulos restantes; si se normalizan textos, se volverán a contar las categorías; si se eliminan filas, se compararán las dimensiones.

In [ ]:
columnas_importantes = [
    "Item",
    "Quantity",
    "Price Per Unit",
    "Total Spent",
    "Transaction Date"
]

diagnostico_faltantes.loc[columnas_importantes]

La selección concentra la atención en las variables que pueden influir directamente en ventas, cantidades, precios y periodos.

La importancia de un faltante depende de dos elementos: su frecuencia y la función de la columna. Una ausencia poco frecuente en una variable esencial puede requerir más cuidado que muchas ausencias en una variable secundaria.

### Criterio de interpretación

Esta primera revisión establece una referencia para las comprobaciones posteriores. Mantener claro el punto de partida facilita comparar qué cambió y qué problemas permanecen después de limpiar.

## Ausencias escritas como texto

Hasta ahora contamos valores que Pandas reconoce como nulos. Sin embargo, las fuentes reales también pueden utilizar palabras para indicar que no hay información o que la carga fue defectuosa.

Algunos ejemplos son `UNKNOWN`, `ERROR`, `N/A`, `None` o cadenas vacías. Si se almacenan como texto, `isna()` no siempre los incluye en el conteo. Por eso revisaremos las categorías de algunas columnas.

### Criterio de interpretación

La ausencia debe interpretarse junto con el significado de la variable. La misma cantidad de valores faltantes puede ser tolerable en una columna descriptiva y crítica en una variable necesaria para calcular resultados.

In [ ]:
df["Item"].value_counts(dropna=False)

In [ ]:
df["Payment Method"].value_counts(dropna=False)

In [ ]:
df["Location"].value_counts(dropna=False)

El argumento `dropna=False` conserva los nulos dentro del conteo de frecuencias.

Al revisar estas listas debemos distinguir entre categorías válidas y marcadores de ausencia o error. Una cadena como `UNKNOWN` puede parecer una categoría más, aunque semánticamente signifique que no se conoce el dato.

Esta inspección amplía el diagnóstico más allá de los `NaN` visibles para Pandas.

### Criterio de interpretación

En una fuente real, el formato de almacenamiento y el significado del dato no siempre coinciden. Por eso conviene observar ejemplos concretos antes de convertir, reemplazar o eliminar valores.

## Diferenciar nulos y marcadores especiales

Aquí aparecen dos fenómenos que se parecen, pero no son idénticos. `NaN` es una ausencia reconocida por Pandas; `UNKNOWN` y `ERROR` son textos que requieren interpretación.

La diferencia importa porque cada grupo puede necesitar un tratamiento distinto. Antes de reemplazar valores, conviene contabilizarlos por columna y verificar si realmente representan ausencia, error o una categoría válida del negocio.

### Criterio de interpretación

El diagnóstico combina evidencia local y global: las filas muestran casos reales, mientras que los conteos y porcentajes permiten evaluar si esos casos son aislados o forman un patrón.

In [ ]:
valores_problematicos = ["UNKNOWN", "ERROR"]

for columna in ["Item", "Payment Method", "Location"]:
    print(f"Columna: {columna}")
    print(df[columna].isin(valores_problematicos).sum())
    print()

`isin()` compara los valores de una columna con una lista de posibilidades y produce una máscara. Al sumar esa máscara obtenemos cuántas veces aparecen los marcadores definidos.

La salida permite contrastar estos conteos con los obtenidos mediante `isna()`. De ese modo podemos estimar si el problema de información faltante está subrepresentado cuando solo buscamos nulos reales.

### Criterio de interpretación

Todavía no se aplica una estrategia de limpieza. Separar medición y transformación ayuda a evitar decisiones irreversibles basadas en una sola muestra o en una interpretación apresurada.

In [ ]:
diagnostico_textos_problematicos = pd.DataFrame({
    "UNKNOWN_o_ERROR": {
        columna: df[columna].isin(valores_problematicos).sum()
        for columna in ["Item", "Payment Method", "Location"]
    }
})

diagnostico_textos_problematicos

La tabla organiza la cantidad de marcadores especiales por variable categórica.

Este registro no transforma el dataset. Su función es dejar evidencia de qué columnas contienen valores que deben interpretarse antes de decidir si se recodifican, se convierten en nulos o se conservan como categorías explícitas.

### Criterio de interpretación

Estas comprobaciones también son útiles para comunicar resultados. Una tabla de diagnóstico permite explicar con claridad qué se encontró, dónde apareció y qué información adicional se necesita.

## Consolidar los hallazgos

Después de revisar nulos, porcentajes y textos especiales, es conveniente reunir la información en tablas de diagnóstico. Esto facilita comparar el estado de las columnas y evita depender únicamente de salidas separadas.

La consolidación también prepara una base para documentar decisiones posteriores: qué problema se observó, en qué variable apareció y qué comprobación deberá realizarse después de cualquier transformación.

### Criterio de interpretación

Cuando una ausencia se concentra en una variable central, conviene estudiar sus consecuencias antes de continuar con cálculos o modelos. La calidad del resultado depende de reconocer esas limitaciones.

In [ ]:
diagnostico_faltantes = pd.DataFrame({
    "faltantes_nan": df.isna().sum(),
    "porcentaje_nan": df.isna().mean() * 100
})

diagnostico_faltantes.sort_values("faltantes_nan", ascending=False)

La primera tabla resume las ausencias detectadas directamente por Pandas. Esa información es necesaria, pero no cubre todos los valores que pueden representar falta de datos.

Por eso se complementará con una revisión de marcadores textuales. Leer ambas partes juntas permite distinguir la ausencia técnica de la ausencia semántica.

### Criterio de interpretación

La revisión de valores especiales complementa el análisis de `NaN`. Un dataset puede representar la falta de información de varias maneras, especialmente cuando proviene de sistemas distintos o de procesos de captura manual.

In [ ]:
columnas_categoricas_revisar = ["Item", "Payment Method", "Location"]

diagnostico_textos = pd.DataFrame({
    "UNKNOWN_o_ERROR": {
        columna: df[columna].isin(["UNKNOWN", "ERROR"]).sum()
        for columna in columnas_categoricas_revisar
    }
})

diagnostico_textos

Las dos tablas describen problemas relacionados, pero diferentes: una cuenta nulos reconocidos y la otra cuenta textos que podrían tener significado de ausencia o error.

Esta separación es útil porque evita mezclar categorías legítimas con valores defectuosos sin evidencia. El siguiente paso será elegir una estrategia específica para cada caso.

### Criterio de interpretación

Ordenar y resumir las evidencias no modifica el contenido de la fuente. Solo mejora la lectura y ayuda a establecer un orden razonable para las siguientes decisiones.

## Por qué el diagnóstico viene primero

Detectar valores faltantes no significa que debamos eliminarlos inmediatamente. Borrar filas puede reducir la cantidad de datos, completar valores puede introducir supuestos y descartar una columna puede eliminar una variable útil.

La decisión depende del propósito del análisis, del porcentaje afectado, de la importancia de la columna y de la posibilidad de reconstruir el dato. En este capítulo solo reunimos evidencia para que esas decisiones puedan tomarse con criterio.

### Criterio de interpretación

La validación posterior deberá responder a la señal encontrada. Si se modifican nulos, se revisarán nulos restantes; si se normalizan textos, se volverán a contar las categorías; si se eliminan filas, se compararán las dimensiones.

## Recapitulación

En este capítulo construimos un diagnóstico de valores faltantes. Comenzamos identificando `NaN`, calculamos conteos y porcentajes, ordenamos los resultados, localizamos filas afectadas y enfocamos la revisión en variables importantes.

También observamos que una fuente puede representar la ausencia mediante textos como `UNKNOWN` o `ERROR`. Por ello, revisar `isna()` no basta: hay que conocer las convenciones de la fuente y comparar los valores categóricos.

La idea central es:

```text
medir primero, interpretar después y transformar solo con una justificación
```

### Criterio de interpretación

Esta primera revisión establece una referencia para las comprobaciones posteriores. Mantener claro el punto de partida facilita comparar qué cambió y qué problemas permanecen después de limpiar.

## Siguiente etapa

La continuación abordará las alternativas para tratar la información faltante. Compararemos cuándo puede ser razonable eliminar registros, cuándo conviene conservarlos, cuándo una imputación es defendible y qué riesgos acompañan a cada opción.

El diagnóstico reunido aquí será la referencia para validar cualquier cambio posterior.

### Criterio de interpretación

La ausencia debe interpretarse junto con el significado de la variable. La misma cantidad de valores faltantes puede ser tolerable en una columna descriptiva y crítica en una variable necesaria para calcular resultados.